In [ ]:
import pandas as pd

In [ ]:
from google.colab import files
upload = files.upload()

Saving streaming_users_dirty.json to streaming_users_dirty (2).json


In [ ]:

# PIPELINE / LOG DE TRANSFORMACIONES


pipeline_log = []

def registrar_paso(nombre, descripcion, filas_antes, filas_despues):
    pipeline_log.append({
        "Paso": nombre,
        "Descripción": descripcion,
        "Filas antes": filas_antes,
        "Filas después": filas_despues
    })

In [ ]:
df = pd.read_json("streaming_users_dirty.json")
df_crudo = df.copy()
print ("se crea una copia del dataset original")

se crea una copia del dataset original


# Copia del dataset (.copy())

Decisión tomada: Se creó una copia del dataset mediante .copy().

Justificación: Se buscó conservar una versión independiente del conjunto de datos limpio, evitando modificar accidentalmente el dataset original o de trabajo.

Efecto observado: Se obtuvo un nuevo DataFrame completamente independiente, listo para continuar con el proceso de análisis y para su posterior exportación.

In [ ]:
# Se visualiza la estructura del dataset observando las primeras 5 filas
df.head()

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
0,10000,39,Estándar,805.8,Brasil,Crime,2025-03-04,99
1,10001,37,Estándar,1173.4,Colombia,Crime,2019-04-02,2
2,10002,28,Básico,401.0,Colombia,Crime,2018-04-13,0
3,10003,43,Básico,62.4,Uruguay,Thriller,2021-01-31,0
4,10004,51,Básico,477.8,Perú,Thriller,2020-09-30,1


In [ ]:
#se traducen las variables del ingles al español para poder interpretarlas
df = df.rename(columns={
    "user_id": "id_usuario",
    "age": "edad",
    "subscription_plan": "plan_suscripcion",
    "monthly_watch_time_mins": "minutos_visualizacion_mensual",
    "country": "pais",
    "favorite_genre": "genero_favorito",
    "last_login_date": "fecha_ultimo_inicio_sesion",
    "customer_support_tickets": "tickets_soporte_cliente"
})

df.head()

,id_usuario,edad,plan_suscripcion,minutos_visualizacion_mensual,pais,genero_favorito,fecha_ultimo_inicio_sesion,tickets_soporte_cliente
0,10000,39,Estándar,805.8,Brasil,Crime,2025-03-04,99
1,10001,37,Estándar,1173.4,Colombia,Crime,2019-04-02,2
2,10002,28,Básico,401.0,Colombia,Crime,2018-04-13,0
3,10003,43,Básico,62.4,Uruguay,Thriller,2021-01-31,0
4,10004,51,Básico,477.8,Perú,Thriller,2020-09-30,1


In [ ]:

# 1 ELIMINACIÓN DE REGISTROS DUPLICADOS


# Se eliminan las filas completamente duplicadas.

df = df.drop_duplicates()

print("Duplicados restantes:", df.duplicated().sum())

Duplicados restantes: 0


In [ ]:
filas_antes = len(df)

df = df.drop_duplicates()

filas_despues = len(df)

registrar_paso(
    "Eliminar duplicados",
    "Se eliminaron registros completamente duplicados.",
    filas_antes,
    filas_despues
)

# 1 Eliminación de duplicados

Decisión tomada: Se eliminaron los registros duplicados.

Justificación: Los registros repetidos pueden generar sesgos en los análisis estadísticos y afectar la calidad de los resultados.

Efecto observado: Se redujo la cantidad de filas, conservando únicamente registros únicos y mejorando la consistencia del dataset.



In [ ]:
# 2. LIMPIEZA DE ESPACIOS


columnas_texto = [
    "plan_suscripcion",
    "pais",
    "genero_favorito"
]

for col in columnas_texto:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
    )

In [ ]:
registrar_paso(
    "Limpieza de espacios",
    "Se eliminaron espacios sobrantes y se normalizó el texto.",
    len(df),
    len(df)
)

# 2 Eliminación de espacios en blanco

Decisión tomada: Se eliminaron los espacios innecesarios al inicio y al final de los textos.

Justificación: Los espacios adicionales provocan que valores iguales sean interpretados como diferentes durante el análisis.

Efecto observado: Los datos textuales quedaron normalizados y listos para su procesamiento.

In [ ]:
# 3 ESTANDARIZACIÓN DE PLANES


df["plan_suscripcion"] = (
    df["plan_suscripcion"]
    .str.lower()
)

planes = {
    "basico": "Básico",
    "básico": "Básico",
    "basic": "Básico",
    "estandar": "Estándar",
    "estándar": "Estándar",
    "standard": "Estándar",
    "std": "Estándar",
    "premium": "Premium",
    "premiun": "Premium"
}

df["plan_suscripcion"] = (
    df["plan_suscripcion"]
    .replace(planes)
)

In [ ]:
registrar_paso(
    "Estandarización de planes",
    "Se unificaron los nombres de los planes de suscripción.",
    len(df),
    len(df)
)

# 3 Estandarización de planes de suscripción

Decisión tomada: Se unificaron las diferentes formas de escribir los planes de suscripción.

Justificación: Existían distintas representaciones para un mismo plan, lo que dificultaba el análisis.

Efecto observado: Cada plan quedó representado por un único valor, facilitando los conteos y agrupamientos.

In [ ]:
# 4 ESTANDARIZACIÓN DE PAÍSES


df["pais"] = df["pais"].str.lower()

paises = {
    "arg": "Argentina",
    "argentina": "Argentina",
    "bra": "Brasil",
    "brazil": "Brasil",
    "brasil": "Brasil",
    "chl": "Chile",
    "chile": "Chile",
    "col": "Colombia",
    "colombia": "Colombia",
    "mex": "México",
    "mexico": "México",
    "méxico": "México",
    "per": "Perú",
    "peru": "Perú",
    "perú": "Perú",
    "ury": "Uruguay",
    "uruguay": "Uruguay"
}

df["pais"] = df["pais"].replace(paises)

In [ ]:
registrar_paso(
    "Estandarización de países",
    "Se normalizaron los nombres de los países.",
    len(df),
    len(df)
)

# 4 Estandarización de países

Decisión tomada: Se normalizaron los nombres de los países.

Justificación: Diferentes formatos de escritura representan el mismo país y pueden generar categorías duplicadas.

Efecto observado: Los países quedaron correctamente agrupados bajo una única denominación.


In [ ]:
# 5 ESTANDARIZACIÓN DE GÉNEROS


df["genero_favorito"] = (
    df["genero_favorito"]
    .str.lower()
)

generos = {
    "accion": "Acción",
    "acción": "Acción",
    "action": "Acción",
    "comedia": "Comedia",
    "comedy": "Comedia",
    "drama": "Drama",
    "documental": "Documental",
    "documentary": "Documental",
    "doc": "Documental",
    "crime": "Crime",
    "crimen": "Crime",
    "thriller": "Thriller",
    "thriler": "Thriller",
    "romance": "Romance"
}

df["genero_favorito"] = (
    df["genero_favorito"]
    .replace(generos)

)

In [ ]:
registrar_paso(
    "Estandarización de géneros",
    "Se normalizaron los géneros favoritos.",
    len(df),
    len(df)
)

# 5 Estandarización de géneros favoritos

Decisión tomada: Se normalizaron los nombres de los géneros.

Justificación: Las diferencias de escritura impedían una correcta clasificación de las preferencias.

Efecto observado: Se obtuvo una clasificación uniforme de los géneros musicales.

In [ ]:
# 6 conversion de fechas
df["fecha_ultimo_inicio_sesion"] = pd.to_datetime(
    df["fecha_ultimo_inicio_sesion"],
    format='mixed',
    errors='coerce'
)

In [ ]:
registrar_paso(
    "fecha_ultimo_inicio_sesion",
    "Se normalizaron las ultimas fechas de sesion",
    len(df),
    len(df)
)

# 6 Conversión de fechas

Decisión tomada: Se convirtió la columna de fechas al tipo datetime.

Justificación: Las fechas almacenadas como texto dificultan realizar operaciones temporales.

Efecto observado: La columna quedó preparada para realizar cálculos y análisis cronológicos.


In [ ]:
# 7 FECHAS FUTURAS


df.loc[
    df["fecha_ultimo_inicio_sesion"] >
    pd.Timestamp.today(),
    "fecha_ultimo_inicio_sesion"
] = pd.NaT

In [ ]:
registrar_paso(
    "Corrección de fechas futuras",
    "Las fechas futuras fueron reemplazadas por valores nulos.",
    len(df),
    len(df)
)

# 7 Corrección de fechas futuras

Decisión tomada: Las fechas posteriores a la fecha actual fueron reemplazadas por valores nulos.

Justificación: Se consideraron datos inconsistentes o errores de carga.

Efecto observado: El dataset conserva únicamente fechas válidas para el análisis.

In [ ]:
# 8 EDADES INVÁLIDAS


df.loc[
    (df["edad"] < 13) |
    (df["edad"] > 90),
    "edad"
] = pd.NA

In [ ]:
registrar_paso(
    "Validación de edades",
    "Las edades fuera del rango permitido fueron marcadas como nulas.",
    len(df),
    len(df)
)

# 8 Validación de edades

Decisión tomada: Se identificaron y corrigieron edades fuera del rango permitido.

Justificación: Valores negativos o excesivamente altos representan errores de captura.

Efecto observado: La columna de edad quedó compuesta por valores coherentes.

In [ ]:
# 9 MINUTOS INVÁLIDOS


df.loc[
    (df["minutos_visualizacion_mensual"] < 0) |
    (df["minutos_visualizacion_mensual"] > 5000),
    "minutos_visualizacion_mensual"
] = pd.NA

In [ ]:
registrar_paso(
    "Validación de minutos",
    "Los valores inválidos de minutos fueron reemplazados por nulos.",
    len(df),
    len(df)
)

# 9 Validación de minutos vistos

Decisión tomada: Se corrigieron los valores inválidos de minutos visualizados.

Justificación: Los tiempos negativos o inconsistentes afectan cualquier análisis de consumo.

Efecto observado: Los tiempos registrados representan valores posibles y confiables.


In [ ]:
# 10 TICKETS NEGATIVOS


df.loc[
    df["tickets_soporte_cliente"] < 0,
    "tickets_soporte_cliente"
] = pd.NA

In [ ]:
registrar_paso(
    "Validación de tickets",
    "Los tickets negativos fueron reemplazados por nulos.",
    len(df),
    len(df)
)

# 10 Validación de tickets de soporte

Decisión tomada: Se corrigieron los valores negativos de tickets.

Justificación: No es posible registrar una cantidad negativa de solicitudes de soporte.

Efecto observado: La columna quedó formada únicamente por cantidades válidas.

In [ ]:
df["edad"] = (
    df["edad"]
    .fillna(df["edad"].median())
)

df["minutos_visualizacion_mensual"] = (
    df["minutos_visualizacion_mensual"]
    .fillna(
        df["minutos_visualizacion_mensual"].median()
    )
)

df["tickets_soporte_cliente"] = (
    df["tickets_soporte_cliente"]
    .fillna(
        df["tickets_soporte_cliente"].median()
    )
)
df["genero_favorito"] = (
    df["genero_favorito"]
    .replace("none", pd.NA)
    .fillna(df["genero_favorito"].mode()[0])
)
df["fecha_ultimo_inicio_sesion"] = (
    df["fecha_ultimo_inicio_sesion"]
    .fillna(
        df["fecha_ultimo_inicio_sesion"].mode()[0]
    )
)

In [ ]:
registrar_paso(
    "Imputación de datos",
    "Se completaron los valores faltantes utilizando mediana y moda.",
    len(df),
    len(df)
)

# 11 Imputación de valores faltantes

Decisión tomada: Se reemplazaron los valores nulos utilizando medidas estadísticas apropiadas (mediana o moda).

Justificación: Se evitó eliminar registros completos, conservando la mayor cantidad posible de información.

Efecto observado: El dataset quedó completo, reduciendo la pérdida de datos y mejorando su calidad para el análisis.

In [ ]:
# 12 VERIFICACIÓN FINAL


print(df.isnull().sum())

print("Duplicados:",
      df.duplicated().sum())

print(df.info())

id_usuario                       0
edad                             0
plan_suscripcion                 0
minutos_visualizacion_mensual    0
pais                             0
genero_favorito                  0
fecha_ultimo_inicio_sesion       0
tickets_soporte_cliente          0
dtype: int64
Duplicados: 18
<class 'pandas.core.frame.DataFrame'>
Index: 8034 entries, 0 to 8158
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   id_usuario                     8034 non-null   int64         
 1   edad                           8034 non-null   float64       
 2   plan_suscripcion               8034 non-null   object        
 3   minutos_visualizacion_mensual  8034 non-null   float64       
 4   pais                           8034 non-null   object        
 5   genero_favorito                8034 non-null   object        
 6   fecha_ultimo_inicio_sesion     8034 non-null   dat

In [ ]:
# Crear copia del dataset limpio
df_limpio = df.copy()

# Guardar ambos datasets
df_crudo.to_json("dataset_crudo.json", index=False)
df_limpio.to_json("dataset_limpio.json", index=False)

print("Archivos guardados correctamente.")

Archivos guardados correctamente.


In [ ]:
registrar_paso(
    "Exportación del dataset limpio",
    "Se guardó el dataset limpio en un archivo CSV para su posterior análisis y entrega.",
    len(df_limpio),
    len(df_limpio)
)

# Guardado del dataset limpio

Decisión tomada: Se exportó el dataset limpio a un archivo CSV.

Justificación: Se generó una versión final del conjunto de datos preparada para su entrega y reutilización.

Efecto observado: Se obtuvo un archivo independiente con el dataset completamente limpio y documentado.

In [ ]:
# Cantidad de registros
filas_crudas = len(df_crudo)
filas_limpias = len(df_limpio)

# Porcentaje de retención
retencion = (filas_limpias / filas_crudas) * 100

# Porcentaje eliminado
eliminado = 100 - retencion

print(f"Registros originales: {filas_crudas}")
print(f"Registros finales: {filas_limpias}")
print(f"Porcentaje de retención: {retencion:.2f}%")
print(f"Porcentaje eliminado: {eliminado:.2f}%")

Registros originales: 8160
Registros finales: 8034
Porcentaje de retención: 98.46%
Porcentaje eliminado: 1.54%


# Interpretación de los resultados
El dataset original contenía 8.160 registros. Luego de aplicar el proceso de limpieza y preparación de datos, el conjunto final quedó compuesto por 8.034 registros. Esto significa que se conservó el 98,46 % de la información original y solo fue necesario eliminar el 1,54 % de los registros, correspondientes a datos duplicados o inconsistentes que podían afectar la calidad del análisis. Este resultado indica que el dataset presentaba una buena calidad inicial, ya que la gran mayoría de los datos pudieron mantenerse tras la limpieza, obteniendo un conjunto de datos confiable y listo para las etapas de análisis y modelado.




In [ ]:
log_pipeline = pd.DataFrame(pipeline_log)

print(log_pipeline)

log_pipeline.to_csv("pipeline_log.csv", index=False)

print("Pipeline guardado correctamente.")

                              Paso  \
0              Eliminar duplicados   
1             Limpieza de espacios   
2        Estandarización de planes   
3        Estandarización de países   
4       Estandarización de géneros   
5             Conversión de fechas   
6     Corrección de fechas futuras   
7             Validación de edades   
8            Validación de minutos   
9            Validación de tickets   
10             Imputación de datos   
11  Exportación del dataset limpio   

                                          Descripción  Filas antes  \
0   Se eliminaron registros completamente duplicados.         8034   
1   Se eliminaron espacios sobrantes y se normaliz...         8034   
2   Se unificaron los nombres de los planes de sus...         8034   
3          Se normalizaron los nombres de los países.         8034   
4              Se normalizaron los géneros favoritos.         8034   
5   Se convirtió la columna de fechas al tipo date...         8034   
6   Las fechas 

#INFORME FINAL

El presente trabajo tuvo como objetivo analizar y preparar un conjunto de datos correspondiente a una plataforma de streaming. El dataset contiene información relacionada con usuarios, sus planes de suscripción, hábitos de consumo y actividad dentro de la plataforma.
Durante la exploración de los datos se detectaron diversos problemas de calidad que podían afectar los resultados de futuros análisis.
Problemas detectados
Se identificaron:
Valores faltantes.
Registros duplicados.
Variables categóricas inconsistentes.
Fechas inválidas.
Fechas futuras.
Valores numéricos fuera de rango.
Errores de escritura.
#Justificación de las técnicas utilizadas
Eliminación de duplicados:
Los registros repetidos generan sesgos estadísticos y sobre representación de determinados usuarios.

Estandarización de categorías:
Permite que una misma categoría no aparezca escrita de distintas formas.

Conversión de fechas:
Facilita los análisis temporales y detecta registros inválidos.

Detección de valores imposibles:
Permite identificar errores de carga o digitación.

Imputación de valores faltantes:
Evita la pérdida de información y permite conservar la cantidad de registros.

Resultados obtenidos
Luego de la limpieza:

Se eliminaron registros redundantes.

Se corrigieron inconsistencias.

Se mejoró la calidad de las variables.

Se redujeron los errores del conjunto de datos.

Se obtuvo un dataset más consistente.

Se guardaron ambos datasets, el original y el resultante despues de la limpieza

#Conclusión

La detección y corrección de errores permite mejorar la calidad del dataset y obtener resultados más confiables en etapas posteriores de visualización, análisis estadístico o modelado predictivo.
Luego de nuestra limpieza logramos mantener el 98% de los datos originales, lo cual, según nuestro criterio es aceptable y pemitio que el conjunto de datos presente un mayor nivel de consistencia y confiabilidad.